In [ ]:
from datasets import load_dataset

# Load the "full" configuration and split
dataset = load_dataset(
    "LeMaterial/LeMat-Synth-Papers",
    "full",  # configuration/subset
    split=None,  # other splits include 'arxiv', 'omg24', 'chemrxiv'
    token=True,
)

# Print column names
print(dataset.column_names)

In [ ]:
# Query function, takes in the split name, which column to check and a list of keywords to return the IDs of the papers that match
def query_db(text_column, split_name, list_keywords, exclude_keywords=None):
    ds = dataset[split_name]

    results = {}  # store keyword and corresponding IDs

    for keyword in list_keywords:
        # define filter function for this keyword
        def contains_keyword(example, kw=keyword):
            return (
                example[text_column] is not None
                and kw.lower() in example[text_column].lower()
            )

        # filter dataset
        filtered = ds.filter(contains_keyword)

        if exclude_keywords is not None:

            def contains_exclude_keyword(example):
                return any(
                    ek.lower() in example[text_column].lower()
                    for ek in exclude_keywords
                )

            filtered = filtered.filter(
                lambda x: not contains_exclude_keyword(x)
            )

        # extract IDs
        ids = filtered["id"]

        # store in dictionary
        results[keyword] = ids

    # Print results
    for kw, ids in results.items():
        print(f"\nKeyword: {kw}")
        print(f"Found {len(ids)} entries")
        print(ids)

    return results

In [ ]:
# From a given subset of papers, removes redundancies
def return_nonredundant_ids(results):
    all_ids = []

    for kw, ids in results.items():
        all_ids.extend(ids)

    # Convert to a set to remove duplicates
    unique_ids = set(all_ids)

    # Count them
    print("Number of papers containing at least one keyword:", len(unique_ids))

    return unique_ids

In [ ]:
import pickle

split_name_list = ["arxiv", "omg24", "chemrxiv"]
text_column = "abstract"
list_keywords = [
    "heterogeneous catalysis",
    "heterogeneous catalyst",
    "was heated at",
    "was heated under",
    "thermal treatment",
    "was measured at different temperatures",
    "activation energy",
    "variations",
    "conversion",
    "efficiency",
    "activation energy",
    "catalyst",
    "catalysis",
    "catalytic",
    "TOF",
    "activation energy",
]
exclude_keywords = [
    "electrocatalysis",
    "electrocatalytic",
    "photoelectrocatalysis",
    "photoelectrocatalytic",
    "photocatalysis",
    "photocatalytic",
    "graphene",
]
db = {}

for split_name in split_name_list:
    result = query_db(
        split_name=split_name,
        text_column=text_column,
        list_keywords=list_keywords,
        exclude_keywords=exclude_keywords,
    )
    nonredundant_ids = return_nonredundant_ids(result)
    print(split_name, len(nonredundant_ids), nonredundant_ids)

    db[split_name] = nonredundant_ids

In [ ]:
# export db to pickle file
with open("db_thermocatalysis.pkl", "wb") as f:
    pickle.dump(db, f)

print("Saved db.pkl with keys:", list(db.keys()))

In [ ]:
# Load the filtered IDs from pickle
import pickle

with open("db_thermocatalysis.pkl", "rb") as f:
    db = pickle.load(f)

print("Loaded db with keys:", list(db.keys()))
for split_name, ids in db.items():
    print(f"  {split_name}: {len(ids)} papers")

In [ ]:
# Create the filtered dataset for the new split "thermocatalysis_keywords_only"
from datasets import concatenate_datasets

# Collect all filtered entries from each split
filtered_datasets = []

for split_name in ["arxiv", "omg24", "chemrxiv"]:
    ids_to_keep = db[split_name]
    ds = dataset[split_name]

    # Filter to keep only the IDs in our thermocatalysis set
    filtered = ds.filter(lambda x: x["id"] in ids_to_keep)
    filtered_datasets.append(filtered)
    print(f"{split_name}: kept {len(filtered)} / {len(ds)} papers")

# Concatenate all filtered datasets into one
thermocatalysis_dataset = concatenate_datasets(filtered_datasets)
print(f"\nTotal thermocatalysis papers: {len(thermocatalysis_dataset)}")

In [ ]:
# Create a PR on Hugging Face to add the new split
# Subset: "full", Split: "thermocatalysis_keywords_only"
from huggingface_hub import HfApi

api = HfApi()

# Create a commit on a new branch via PR
thermocatalysis_dataset.push_to_hub(
    repo_id="LeMaterial/LeMat-Synth-Papers",
    config_name="full",  # subset name
    split="thermocatalysis_keywords_only",  # split name
    token=True,
    create_pr=True,  # This creates a PR instead of pushing directly
    commit_message="Add thermocatalysis_keywords_only split with keyword-filtered papers",
)

print(
    "Created PR for subset='full', split='thermocatalysis_keywords_only' on Hugging Face!"
)